In [2]:
import os
import re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from scipy.signal import butter, filtfilt
from sklearn.metrics import precision_recall_curve, auc
import logging

from mamba_ssm import Mamba

# ==========================================
# Global configuration (same as training)
# ==========================================
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

if not torch.cuda.is_available():
    raise RuntimeError("GPU not available, required for visualization")

device = torch.device("cuda")

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = True

TARGET_LEN = 6000
INPUT_CHANNEL = 3
OUTPUT_CHANNEL = 3
FS = 100.0
PICK_TOLERANCE = 50

PICK_THRESHOLD_P = 0.02
PICK_THRESHOLD_S = 0.08

BATCH_SIZE = 32
NOISE_BAND = [1, 45]

OUTPUT_DIR = "./outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)
MODEL_PATH = os.path.join(OUTPUT_DIR, "model_best.pt")
OLD_LOG_PATH = os.path.join(OUTPUT_DIR, "training.log")
REAL_NOISE_PATH = "./stead_real_noise_50000.npy"

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(message)s',
    handlers=[logging.StreamHandler()]
)
logger = logging.getLogger(__name__)

# ==========================================
# Helper functions (identical to training)
# ==========================================
def butter_bandpass(lowcut, highcut, fs, order=4):
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    b, a = butter(order, [low, high], btype='band')
    return b, a

def bandpass_filter(data, lowcut, highcut, fs, order=4):
    b, a = butter_bandpass(lowcut, highcut, fs, order=order)
    y = filtfilt(b, a, data, axis=-1)
    return y

def generate_bandpass_noise(length, fs, band, order=4):
    """Synthetic bandpass noise, not applied to input waveforms"""
    lowcut, highcut = band
    white_noise = np.random.randn(length)
    band_noise = bandpass_filter(white_noise, lowcut, highcut, fs, order)
    return band_noise

def add_noise_with_snr(signal, snr_db, fs, noise_band):
    signal_noisy = np.zeros_like(signal)
    for ch in range(signal.shape[1]):
        sig = signal[:, ch]
        sig_power = np.mean(sig ** 2)
        noise_power = sig_power / (10 ** (snr_db / 10))
        noise = generate_bandpass_noise(len(sig), fs, noise_band)
        noise_scaled = noise * np.sqrt(noise_power / (np.mean(noise ** 2) + 1e-10))
        signal_noisy[:, ch] = sig + noise_scaled
    return signal_noisy

def shift_waveform_and_label(waveform, label, max_shift=500):
    shift = np.random.randint(-max_shift, max_shift)
    shifted_wave = np.zeros_like(waveform)
    shifted_label = np.zeros_like(label)
    if shift > 0:
        shifted_wave[shift:, :] = waveform[:-shift, :]
        shifted_label[shift:, :] = label[:-shift, :]
    elif shift < 0:
        shifted_wave[:shift, :] = waveform[-shift:, :]
        shifted_label[:shift, :] = label[-shift:, :]
    else:
        shifted_wave = waveform
        shifted_label = label
    return shifted_wave, shifted_label, shift

def normalize_single_sample(x):
    """Channel-wise standardization, no filtering"""
    x_normalized = np.zeros_like(x, dtype=np.float32)
    for ch in range(INPUT_CHANNEL):
        mean = np.mean(x[:, ch])
        std = np.std(x[:, ch])
        if std < 1e-6:
            std = 1.0
        x_normalized[:, ch] = (x[:, ch] - mean) / std
    return x_normalized

def generate_triangle_label(length, peak_idx, width=40):
    label = np.zeros(length, dtype=np.float32)
    half_width = width // 2
    start = max(0, peak_idx - half_width)
    end = min(length, peak_idx + half_width)
    rise_len = peak_idx - start
    if rise_len > 0:
        label[start:peak_idx] = np.linspace(0, 1, rise_len)
    label[peak_idx] = 1.0
    fall_len = end - peak_idx
    if fall_len > 0:
        label[peak_idx+1:end] = np.linspace(1, 0, fall_len-1)
    return label

def generate_3channel_labels(y_2channel, width_p=20, width_s=40):
    """Generate soft labels: P width 20, S width 40"""
    n_samples, n_timesteps = y_2channel.shape[0], y_2channel.shape[1]
    y_3channel = np.zeros((n_samples, n_timesteps, 3), dtype=np.float32)
    for i in range(n_samples):
        p_idx = np.argmax(y_2channel[i, :, 0])
        s_idx = np.argmax(y_2channel[i, :, 1])
        y_3channel[i, :, 0] = generate_triangle_label(n_timesteps, p_idx, width=width_p)
        y_3channel[i, :, 1] = generate_triangle_label(n_timesteps, s_idx, width=width_s)
    y_3channel[..., 2] = 1.0 - y_3channel[..., 0] - y_3channel[..., 1]
    return np.clip(y_3channel, 0.0, 1.0)

def calculate_snr(waveform, p_idx, s_idx):
    signal_start = max(0, p_idx - 100)
    signal_end = min(len(waveform), s_idx + 200)
    noise_start = max(0, signal_start - 500)
    noise_end = signal_start
    if noise_end <= noise_start:
        return 0.0
    signal_power = np.mean(waveform[signal_start:signal_end, 2] ** 2)
    noise_power = np.mean(waveform[noise_start:noise_end, 2] ** 2)
    if noise_power < 1e-10:
        return 100.0
    return 10 * np.log10(signal_power / noise_power)

# ==========================================
# Optimized model (num_heads=4, kernel=13, no redundant pre_norm)
# ==========================================
class StochasticDepthDropout(nn.Module):
    def __init__(self, drop_rate):
        super().__init__()
        self.drop_rate = drop_rate

    def forward(self, layer_output, residual):
        if self.training:
            keep_prob = 1 - self.drop_rate
            if torch.rand(1).item() < keep_prob:
                return residual + (layer_output / keep_prob)
            else:
                return residual
        else:
            return residual + layer_output

class DilatedConvBlock(nn.Module):
    """Two dilated conv layers with residual connection."""
    def __init__(self, in_channels, filters, kernel_size=13, dilation_rates=[1, 2], dropout=0.1):
        super().__init__()
        self.conv1 = nn.Conv1d(in_channels, filters, kernel_size,
                               dilation=dilation_rates[0], padding='same')
        self.norm1 = nn.LayerNorm(filters)
        self.act1 = nn.ReLU()
        self.conv2 = nn.Conv1d(filters, filters, kernel_size,
                               dilation=dilation_rates[1], padding='same')
        self.norm2 = nn.LayerNorm(filters)
        self.act2 = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.residual_conv = nn.Conv1d(in_channels, filters, 1) if in_channels != filters else nn.Identity()

    def forward(self, x):
        residual = self.residual_conv(x)
        x = self.conv1(x)
        x = self.norm1(x.permute(0, 2, 1)).permute(0, 2, 1)
        x = self.act1(x)
        x = self.conv2(x)
        x = self.norm2(x.permute(0, 2, 1)).permute(0, 2, 1)
        x = self.act2(x)
        x = self.dropout(x)
        return x + residual

class DualMambaAttentionBlock(nn.Module):
    """Multi-head attention + bidirectional Mamba (optimized, no redundant pre_norm)."""
    def __init__(self, channels, num_heads=4, dropout=0.1, drop_rate=0.0):
        super().__init__()
        self.channels = channels
        self.pre_conv = nn.Sequential(
            nn.Conv1d(channels, channels, kernel_size=13, padding='same'),
            nn.ReLU()
        )
        self.mha_norm = nn.LayerNorm(channels)
        self.mha = nn.MultiheadAttention(
            embed_dim=channels,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True
        )
        self.sdd_mha = StochasticDepthDropout(drop_rate)
        self.mamba_fwd = Mamba(d_model=channels, d_state=8, d_conv=2, expand=2)
        self.mamba_bwd = Mamba(d_model=channels, d_state=8, d_conv=2, expand=2)
        self.fusion = nn.Linear(2 * channels, channels)
        self.mamba_norm = nn.LayerNorm(channels)
        self.sdd_mamba = StochasticDepthDropout(drop_rate)

    def forward(self, x):
        residual_att = x
        x = self.pre_conv(x)
        x = x.permute(0, 2, 1)
        x_norm = self.mha_norm(x)
        att_out, _ = self.mha(x_norm, x_norm, x_norm)
        att_out = att_out.permute(0, 2, 1)
        x = self.sdd_mha(att_out, residual_att)

        residual_mamba = x
        temp_x = x.permute(0, 2, 1)
        out_fwd = self.mamba_fwd(temp_x)
        temp_x_rev = torch.flip(temp_x, dims=[1])
        out_bwd = self.mamba_bwd(temp_x_rev)
        out_bwd = torch.flip(out_bwd, dims=[1])
        out_combined = torch.cat([out_fwd, out_bwd], dim=-1)
        temp_x = self.fusion(out_combined)
        temp_x = self.mamba_norm(temp_x)
        temp_x = temp_x.permute(0, 2, 1)
        x = self.sdd_mamba(temp_x, residual_mamba)
        return x

class EncoderBlock(nn.Module):
    """Dilated conv block + downsampling."""
    def __init__(self, in_channels, out_channels, dilation_rates, dropout=0.1):
        super().__init__()
        self.dilated = DilatedConvBlock(in_channels, out_channels, 13, dilation_rates, dropout)
        self.down = nn.Conv1d(out_channels, out_channels, 13, stride=2, padding=6)

    def forward(self, x):
        skip = self.dilated(x)
        down = F.relu(self.down(skip))
        return skip, down

class DecoderBlock(nn.Module):
    """Upsampling + skip connection + two conv layers."""
    def __init__(self, in_channels, out_channels, skip_channels):
        super().__init__()
        self.up = nn.ConvTranspose1d(in_channels, out_channels, kernel_size=13,
                                     stride=2, padding=6, output_padding=1)
        conv_in = out_channels + skip_channels
        self.conv = nn.Sequential(
            nn.Conv1d(conv_in, out_channels, 13, padding=6),
            nn.ReLU(),
            nn.Conv1d(out_channels, out_channels, 13, padding=6),
            nn.ReLU()
        )

    def forward(self, x, skip):
        x = self.up(x)
        x = F.relu(x)
        x = torch.cat([x, skip], dim=1)
        x = self.conv(x)
        return x

class PhaseNet_Custom_DoubleMamba(nn.Module):
    """U-Net with dual Mamba bottleneck, num_heads=4."""
    def __init__(self, num_heads=4):
        super().__init__()
        self.enc1 = EncoderBlock(INPUT_CHANNEL, 16, [1, 2])
        self.enc2 = EncoderBlock(16, 32, [3, 4])
        self.enc3 = EncoderBlock(32, 64, [5, 6])
        self.enc4 = EncoderBlock(64, 128, [7, 8])

        self.dma1 = DualMambaAttentionBlock(128, num_heads=num_heads, dropout=0.1, drop_rate=0.0)
        self.dma2 = DualMambaAttentionBlock(128, num_heads=num_heads, dropout=0.1, drop_rate=0.1)

        self.dec4 = DecoderBlock(128, 128, 128)
        self.dec3 = DecoderBlock(128, 64, 64)
        self.dec2 = DecoderBlock(64, 32, 32)
        self.dec1 = DecoderBlock(32, 16, 16)

        self.final_conv = nn.Conv1d(16, 3, 1, padding='same')
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, (nn.Conv1d, nn.ConvTranspose1d, nn.Linear)):
                nn.init.kaiming_normal_(m.weight, mode='fan_in', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)

    def forward(self, x):
        x = x.permute(0, 2, 1)            # (B, C, T)

        skip1, x = self.enc1(x)
        skip2, x = self.enc2(x)
        skip3, x = self.enc3(x)
        skip4, x = self.enc4(x)

        x = self.dma1(x)
        x = self.dma2(x)

        x = self.dec4(x, skip4)
        x = self.dec3(x, skip3)
        x = self.dec2(x, skip2)
        x = self.dec1(x, skip1)

        x = self.final_conv(x)
        x = x.permute(0, 2, 1)            # (B, T, C)
        x = torch.softmax(x, dim=-1)
        return x

# ==========================================
# Dataset (no augmentation, only normalization)
# ==========================================
class AugmentedDataset(Dataset):
    def __init__(self, X, y, is_aug=True):
        self.X = X
        self.y = y
        self.is_aug = is_aug

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x_raw = self.X[idx].copy()
        y = self.y[idx].copy()
        # Inference: no augmentation, only normalization
        x_norm = normalize_single_sample(x_raw)
        return torch.tensor(x_norm, dtype=torch.float32), torch.tensor(y, dtype=torch.float32)

# ==========================================
# Metrics (same as training)
# ==========================================
def calc_metrics(y_true, y_pred):
    earthquake_mask = np.max(y_true[..., 0], axis=1) > 0.5
    noise_mask = ~earthquake_mask
    y_true_eq = y_true[earthquake_mask]
    y_pred_eq = y_pred[earthquake_mask]
    y_true_noise = y_true[noise_mask]
    y_pred_noise = y_pred[noise_mask]
    n_eq = len(y_true_eq)
    n_noise = len(y_true_noise)
    metrics = {}

    p_errors_list = []
    s_errors_list = []

    for ch, name in enumerate(['P', 'S']):
        true_idx = np.argmax(y_true_eq[..., ch], axis=1)
        pred_max_prob = np.max(y_pred_eq[..., ch], axis=1)
        pred_idx = np.argmax(y_pred_eq[..., ch], axis=1)

        if name == 'P':
            valid_detect = pred_max_prob >= PICK_THRESHOLD_P
        else:
            valid_detect = pred_max_prob >= PICK_THRESHOLD_S

        valid_idx = np.where(valid_detect)[0]
        sample_errors = (pred_idx[valid_idx] - true_idx[valid_idx]) / FS
        if name == 'P':
            p_errors_list = sample_errors
        else:
            s_errors_list = sample_errors

        pick_error = np.abs(true_idx - pred_idx)
        correct = valid_detect & (pick_error <= PICK_TOLERANCE)
        tp = np.sum(correct)
        fn = n_eq - tp
        mae = np.mean(pick_error[correct]) if tp > 0 else np.inf

        noise_max_prob = np.max(y_pred_noise[..., ch], axis=1)
        if name == 'P':
            fp = np.sum(noise_max_prob >= PICK_THRESHOLD_P)
        else:
            fp = np.sum(noise_max_prob >= PICK_THRESHOLD_S)
        tn = n_noise - fp

        precision = tp / (tp + fp + 1e-6)
        recall = tp / (tp + fn + 1e-6)
        f1 = 2 * (precision * recall) / (precision + recall + 1e-6)

        y_true_binary = np.concatenate([np.ones(n_eq), np.zeros(n_noise)])
        y_pred_score = np.concatenate([np.max(y_pred_eq[..., ch], axis=1), np.max(y_pred_noise[..., ch], axis=1)])
        prec_curve, rec_curve, _ = precision_recall_curve(y_true_binary, y_pred_score)
        auc_score = auc(rec_curve, prec_curve)

        metrics[f'mae_{name}'] = mae
        metrics[f'tp_{name}'] = tp
        metrics[f'fn_{name}'] = fn
        metrics[f'fp_{name}'] = fp
        metrics[f'tn_{name}'] = tn
        metrics[f'prec_{name}'] = precision
        metrics[f'rec_{name}'] = recall
        metrics[f'f1_{name}'] = f1
        metrics[f'auc_{name}'] = auc_score

    metrics['n_earthquake'] = n_eq
    metrics['n_noise'] = n_noise
    return metrics, np.array(p_errors_list), np.array(s_errors_list)

# ==========================================
# Extract loss from training log
# ==========================================
def extract_and_save_loss_log():
    logger.info("\nExtracting loss from training log...")
    loss_log_path = os.path.join(OUTPUT_DIR, "loss_log.txt")
    if not os.path.exists(OLD_LOG_PATH):
        logger.warning(f"Training log not found: {OLD_LOG_PATH}, skipping.")
        return
    try:
        with open(OLD_LOG_PATH, 'r', encoding='utf-8') as f:
            log_lines = f.readlines()
        loss_data = []
        pattern = re.compile(r'Train Loss: ([\d.]+) \| Val Loss: ([\d.]+)')
        epoch_counter = 1
        for line in log_lines:
            match = pattern.search(line)
            if match:
                train_loss = match.group(1)
                val_loss = match.group(2)
                loss_data.append(f"Epoch {epoch_counter}: Train Loss = {train_loss}, Val Loss = {val_loss}")
                epoch_counter += 1
        if loss_data:
            with open(loss_log_path, 'w', encoding='utf-8') as f:
                f.write("Training Loss Log\n")
                f.write("="*50 + "\n")
                for line in loss_data:
                    f.write(line + "\n")
            logger.info(f"Loss log saved to: {loss_log_path}")
        else:
            logger.warning("No loss data found in log.")
    except Exception as e:
        logger.warning(f"Error extracting loss: {e}")

# ==========================================
# Main evaluation
# ==========================================
def main():
    logger.info("="*80)
    logger.info("  Model evaluation script | Optimized architecture (kernel=13, heads=4)")
    logger.info("="*80)

    # 1. Extract loss log
    extract_and_save_loss_log()

    # 2. Load dataset and split (7:2:1 as in training)
    logger.info("\nLoading STEAD dataset...")
    try:
        X_original = np.load("chunk2_stead_X.npy").astype(np.float32)
        y_2channel = np.load("chunk2_stead_y.npy").astype(np.float32)
        all_real_noise = np.load(REAL_NOISE_PATH).astype(np.float32)
    except FileNotFoundError as e:
        logger.error(f"Data file not found: {e}")
        return

    logger.info("Reproducing train/val/test split (7:2:1)...")
    indices = np.arange(len(X_original))
    np.random.shuffle(indices)
    n_total_eq = len(X_original)
    n_train_eq = int(n_total_eq * 0.7)
    n_val_eq = int(n_total_eq * 0.2)

    test_eq_indices = indices[n_train_eq + n_val_eq:]
    X_test_eq = X_original[test_eq_indices]
    y_test_2ch = y_2channel[test_eq_indices]
    y_test_eq = generate_3channel_labels(y_test_2ch, width_p=20, width_s=40)

    logger.info("Loading real noise test set...")
    all_real_noise = all_real_noise[:50000]
    n_train_noise = int(50000 * 0.7)
    n_val_noise = int(50000 * 0.2)
    n_test_noise = 50000 - n_train_noise - n_val_noise

    noise_indices = np.arange(50000)
    np.random.shuffle(noise_indices)
    test_noise_indices = noise_indices[n_train_noise + n_val_noise:]

    X_test_noise = all_real_noise[test_noise_indices]
    y_test_noise = np.zeros((n_test_noise, TARGET_LEN, 3), dtype=np.float32)
    y_test_noise[..., 2] = 1.0

    X_test_final = np.concatenate([X_test_eq, X_test_noise], axis=0)
    y_test_final = np.concatenate([y_test_eq, y_test_noise], axis=0)
    logger.info(f"Test set: events={len(X_test_eq)}, noise={len(X_test_noise)}, total={len(X_test_final)}")

    # 3. Load model (num_heads=4)
    logger.info(f"\nLoading optimal model: {MODEL_PATH}...")
    if not os.path.exists(MODEL_PATH):
        logger.error(f"Model file not found: {MODEL_PATH}")
        return
    model = PhaseNet_Custom_DoubleMamba(num_heads=4).to(device)
    model.load_state_dict(torch.load(MODEL_PATH, map_location=device, weights_only=True))
    model.eval()
    logger.info("Model loaded successfully")

    # 4. Inference
    logger.info("\nRunning inference on test set...")
    dataset = AugmentedDataset(X_test_final, y_test_final, is_aug=False)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
    all_preds = []
    with torch.no_grad():
        for batch in tqdm(loader, desc="Inferencing"):
            x, _ = batch
            x = x.to(device)
            pred = model(x)
            all_preds.append(pred.cpu().numpy())
    all_preds = np.concatenate(all_preds, axis=0)
    logger.info("Inference complete")

    # 5. Compute and print metrics
    logger.info("\n" + "="*80)
    logger.info("  Test Set Metrics")
    logger.info("="*80)
    m, p_errors, s_errors = calc_metrics(y_test_final, all_preds)

    logger.info(f"\n  {'Metric':<12} | {'P-wave':<12} | {'S-wave':<12}")
    logger.info("-"*80)
    logger.info(f"  {'MAE(sec)':<12} | {m['mae_P']/FS:<12.3f} | {m['mae_S']/FS:<12.3f}")
    logger.info(f"  {'PR-AUC':<12} | {m['auc_P']:<12.4f} | {m['auc_S']:<12.4f}")
    logger.info(f"  {'TP':<12} | {m['tp_P']:<12.0f} | {m['tp_S']:<12.0f}")
    logger.info(f"  {'TN':<12} | {m['tn_P']:<12.0f} | {m['tn_S']:<12.0f}")
    logger.info(f"  {'FN':<12} | {m['fn_P']:<12.0f} | {m['fn_S']:<12.0f}")
    logger.info(f"  {'FP':<12} | {m['fp_P']:<12.0f} | {m['fp_S']:<12.0f}")
    logger.info(f"  {'Precision':<12} | {m['prec_P']:<12.4f} | {m['prec_S']:<12.4f}")
    logger.info(f"  {'Recall':<12} | {m['rec_P']:<12.4f} | {m['rec_S']:<12.4f}")
    logger.info(f"  {'F1-score':<12} | {m['f1_P']:<12.4f} | {m['f1_S']:<12.4f}")
    logger.info("="*80)

    # 6. Save results for earthquake events only
    logger.info("\nGenerating result files...")
    earthquake_mask = np.arange(len(X_test_final)) < len(X_test_eq)
    X_test_eq_wave = X_test_final[earthquake_mask]
    y_true_eq = y_test_final[earthquake_mask]
    y_pred_eq = all_preds[earthquake_mask]

    sample_idx_list = test_eq_indices
    itp_true = np.argmax(y_true_eq[..., 0], axis=1)
    its_true = np.argmax(y_true_eq[..., 1], axis=1)
    itp_pred = np.argmax(y_pred_eq[..., 0], axis=1)
    its_pred = np.argmax(y_pred_eq[..., 1], axis=1)
    p_err_sec = (itp_pred - itp_true) / FS
    s_err_sec = (its_pred - its_true) / FS

    npz_path = os.path.join(OUTPUT_DIR, 'test_picking_results.npz')
    np.savez_compressed(
        npz_path,
        sample_index=sample_idx_list,
        waveform=X_test_eq_wave,
        itp_true=itp_true,
        its_true=its_true,
        itp_pred=itp_pred,
        its_pred=its_pred,
        p_error_sec=p_err_sec,
        s_error_sec=s_err_sec,
        sampling_rate=FS,
        pick_threshold_p=PICK_THRESHOLD_P,
        pick_threshold_s=PICK_THRESHOLD_S
    )
    logger.info(f"Results saved to: {npz_path}")

    csv_path = os.path.join(OUTPUT_DIR, 'test_picking_results.csv')
    result_df = pd.DataFrame({
        'sample_id': sample_idx_list,
        'itp_true': itp_true,
        'its_true': its_true,
        'itp_pred': itp_pred,
        'its_pred': its_pred,
        'p_error_sec': np.round(p_err_sec, 4),
        's_error_sec': np.round(s_err_sec, 4),
        'abs_p_error_sec': np.round(np.abs(p_err_sec), 4),
        'abs_s_error_sec': np.round(np.abs(s_err_sec), 4)
    })
    result_df.to_csv(csv_path, index=False)
    logger.info(f"CSV saved to: {csv_path}")

    stats_path = os.path.join(OUTPUT_DIR, 'test_error_stats.txt')
    with open(stats_path, 'w', encoding='utf-8') as f:
        f.write("Test Set Error Statistics\n")
        f.write("="*50 + "\n\n")
        f.write(f"P-wave MAE: {np.mean(np.abs(p_errors)):.3f} s\n")
        f.write(f"P-wave STD: {np.std(p_errors):.3f} s\n")
        f.write(f"P-wave RMSE: {np.sqrt(np.mean(p_errors**2)):.3f} s\n\n")
        f.write(f"S-wave MAE: {np.mean(np.abs(s_errors)):.3f} s\n")
        f.write(f"S-wave STD: {np.std(s_errors):.3f} s\n")
        f.write(f"S-wave RMSE: {np.sqrt(np.mean(s_errors**2)):.3f} s\n")
    logger.info(f"Error statistics saved to: {stats_path}")

    logger.info("\n" + "="*80)
    logger.info("  All tasks completed. Results in outputs folder.")
    logger.info("="*80)

if __name__ == "__main__":
    main()

2026-06-04 23:51:19,039 - ================================================================================
2026-06-04 23:51:19,040 -   Model evaluation script | Optimized architecture (kernel=13, heads=4)
2026-06-04 23:51:19,041 - ================================================================================
2026-06-04 23:51:19,042 - 
Extracting loss from training log...
2026-06-04 23:51:19,045 - Loss log saved to: ./outputs/loss_log.txt
2026-06-04 23:51:19,045 - 
Loading STEAD dataset...
2026-06-04 23:51:45,858 - Reproducing train/val/test split (7:2:1)...
2026-06-04 23:51:48,044 - Loading real noise test set...
2026-06-04 23:51:48,837 - Test set: events=20000, noise=5000, total=25000
2026-06-04 23:51:48,838 - 
Loading optimal model: ./outputs/model_best.pt...
2026-06-04 23:51:48,918 - Model loaded successfully
2026-06-04 23:51:48,919 - 
Running inference on test set...
Inferencing: 100%|████████████████████████████| 782/782 [00:08<00:00, 89.56it/s]
2026-06-04 23:51:58,021 - Inferen